# 5. Evaluate Translation Emotion Preservation with Fine-tuned mBERT model

This notebook applies the best fine-tuned mBERT model from notebook 4 to the English ttranslations produced in nootebook 3.

We do NOT train/tune mBERT or do any prompt engineering to tweek the English translations in this notebook.

## Evaluation idea
Since each SMP post already has a gold emotion label in the source language (Chinese). For each translation we will use mBERT to predict one of the six emotions (note that we'll need to map the label to the shared lables); then compare the English predition to the original Chinese gold emotion label.

## Primary metrics
- **emotion-preservation accuracy** = percentage of translated posts whose predicted English emotion that matches the original SMP gold emotion
- **macro-F1 vs. source gold** = six-class F1 of the translated English predictions against the original source labels

## Translation files used
- `rq1_matched_eval_baseline.csv`
- `rq1_matched_test_baseline.csv`
- `rq2_matched_eval_context_aware.csv`
- `rq2_matched_test_context_aware.csv`
- `rq3_matched_eval_incorrect.csv`
- `rq3_matched_test_incorrect.csv`



## 0. Install dependencies and imports required modules

In [1]:
%pip install -q -U "transformers>=4.45,<6" datasets accelerate scikit-learn statsmodels matplotlib

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.


In [23]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from datasets import Dataset
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score
)
from statsmodels.stats.contingency_tables import mcnemar
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer
)

RANDOM_SEED = 266
BASE_MODEL_NAME = "google-bert/bert-base-multilingual-cased"
MAX_LENGTH = 128

current_folder = Path.cwd()
project_root = (
    current_folder.parent
    if current_folder.name == "notebooks"
    else current_folder
)

data_folder = project_root / "data"
results_folder = project_root / "data" / "results"
figures_folder = results_folder / "figures"

MODEL_DIR = (
    project_root
    / "models"
    / "mbert_emotion_6class"
    / "checkpoints"
    / "checkpoint-7134"
)

results_folder.mkdir(parents=True, exist_ok=True)
figures_folder.mkdir(parents=True, exist_ok=True)

print("Translation folder:", data_folder)

print("Model checkpoint:", MODEL_DIR)
print("Model directory exists:", MODEL_DIR.exists())

print("Results folder:", results_folder)

print("CUDA available:", torch.cuda.is_available())

Translation folder: /content/data
Model checkpoint: /content/models/mbert_emotion_6class/checkpoints/checkpoint-7134
Model directory exists: True
Results folder: /content/data/results
CUDA available: True


### Definte the shared labels
We've done this in notebook 4 but it's more convenient to redefine it here

In [10]:
LABELS = [
    "joy",
    "anger",
    "sadness",
    "fear",
    "surprise",
    "neutral",
]

label_to_id = {label: idx for idx, label in enumerate(LABELS)}
id_to_label = {idx: label for label, idx in label_to_id.items()}

SMP_TO_SHARED = {
    "happy": "joy",
    "angry": "anger",
    "sad": "sadness",
    "fear": "fear",
    "surprise": "surprise",
    "neutral": "neutral",
}

print(label_to_id)

{'joy': 0, 'anger': 1, 'sadness': 2, 'fear': 3, 'surprise': 4, 'neutral': 5}


### Load the final (best) mBERT classifier from notebook 4
I've saved the required files from checkpoint-7134 from notebook 4. We can simple load it without having to retrain the model here

In [21]:
# load model
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)

# load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

loaded_labels = [model.config.id2label[i] for i in range (model.config.num_labels)]
print("Loaded labels:", loaded_labels)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loaded labels: ['joy', 'anger', 'sadness', 'fear', 'surprise', 'neutral']


### Load and validate 6 translation files

Expected fields include:
- `pair_id`
- `split`
- `label`
- `role`
- `post_id`
- `condition`
- `translation`


In [24]:
TRANSLATION_FILES = {
    "rq1_eval": "baseline_translation/rq1_matched_eval_baseline.csv",
    "rq1_test": "baseline_translation/rq1_matched_test_baseline.csv",
    "rq2_eval": "context_aware_translation/rq2_matched_eval_context_aware.csv",
    "rq2_test": "context_aware_translation/rq2_matched_test_context_aware.csv",
    "rq3_eval": "incorrect_definition_translation/rq3_matched_eval_incorrect.csv",
    "rq3_test": "incorrect_definition_translation/rq3_matched_test_incorrect.csv",
}

required_columns = {
    "pair_id",
    "split",
    "label",
    "role",
    "post_id",
    "condition",
    "translation",
}


translation_dfs = {}

for key, filename in TRANSLATION_FILES.items():
    path = data_folder / filename

    if not path.exists():
        raise FileNotFoundError(f"Missing {path}")

    df = pd.read_csv(path, encoding="utf-8-sig")

    missing = required_columns - set(df.columns)
    if missing:
        raise ValueError(f"{filename} is missing columns: {missing}")

    if df["translation"].isna().any():
        raise ValueError(f"{filename} contains missing translations.")

    translation_dfs[key] = df.copy()

    print(
        key,
        "| rows:", len(df),
        "| split:", df["split"].unique().tolist(),
        "| roles:", df["role"].value_counts().to_dict(),
        "| conditions:", df["condition"].value_counts().to_dict(),
    )

rq1_eval | rows: 166 | split: ['eval'] | roles: {'slang': 83, 'control': 83} | conditions: {'baseline': 166}
rq1_test | rows: 166 | split: ['test'] | roles: {'slang': 83, 'control': 83} | conditions: {'baseline': 166}
rq2_eval | rows: 83 | split: ['eval'] | roles: {'slang': 83} | conditions: {'context-aware': 83}
rq2_test | rows: 83 | split: ['test'] | roles: {'slang': 83} | conditions: {'context-aware': 83}
rq3_eval | rows: 83 | split: ['eval'] | roles: {'slang': 83} | conditions: {'incorrect': 83}
rq3_test | rows: 83 | split: ['test'] | roles: {'slang': 83} | conditions: {'incorrect': 83}


In [25]:
# Validate RQ1 pair structure.
for split in ["eval", "test"]:
    rq1 = translation_dfs[f"rq1_{split}"]
    pair_sizes = rq1.groupby("pair_id").size()
    assert (pair_sizes == 2).all()

    role_sets = rq1.groupby("pair_id")["role"].apply(set)
    assert role_sets.apply(lambda roles: roles == {"slang", "control"}).all()

    # Validate that RQ2 and RQ3 use the exact same slang posts.
    rq1_slang = rq1[rq1["role"] == "slang"]

    for rq in ["rq2", "rq3"]:
        other = translation_dfs[f"{rq}_{split}"]

        assert set(rq1_slang["post_id"]) == set(other["post_id"])
        assert set(rq1_slang["pair_id"]) == set(other["pair_id"])

print("All pairing/alignment checks passed.")

All pairing/alignment checks passed.


## 1. Apply the frozen mBERT predictor on the translated posts

For every English translation we save:

- predicted emotion
- prediction confidence
- probability for each of the six emotions
- whether the predicted emotion matches the original SMP gold emotion

In [26]:
def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

inference_trainer = Trainer(
    model=model,
    data_collator=data_collator,
    processing_class=tokenizer,
)


def classify_translation(df):
    output_df = df.copy()
    output_df["gold_shared_label"] = (output_df["label"].map(SMP_TO_SHARED))

    inference_df = pd.DataFrame({"text": output_df["translation"].fillna("").astype(str)})

    dataset = Dataset.from_pandas(inference_df, preserve_index=False)
    tokenized_dataset = dataset.map(tokenize_function, batched=True)

    prediction_output = inference_trainer.predict(tokenized_dataset)

    logits = prediction_output.predictions

    probabilities = torch.softmax(torch.tensor(logits), dim = 1).numpy()

    prediction_ids = probabilities.argmax(axis = 1)

    # write to output df
    output_df["mbert_pred_id"] = prediction_ids
    output_df["mbert_prediction"] = [id_to_label[id] for id in prediction_ids]
    output_df["mbert_confidence"] = probabilities.max(axis = 1)
    output_df["mbert_correct"] = (output_df["mbert_prediction"] == output_df["gold_shared_label"])

    # write the probability for each possible emotion class into a separate columns
    for idx, label in enumerate(LABELS):
        output_df[f"p_{label}"] = probabilities[:, idx]

    return output_df

In [27]:
# Run mBERT on every translation
prediction_dfs = {}

for key, df in translation_dfs.items():
    print(f"Classifying {key}...")
    pred_df = classify_translation(df)

    prediction_dfs[key] = classify_translation(df)

    input_name = TRANSLATION_FILES[key]
    output_name = (Path(input_name).stem + "_mbert_predictions.csv")

    output_path = results_folder / output_name

    pred_df.to_csv(output_path, index=False, encoding="utf-8-sig")

    print("mBERT average classification score: :",f"{pred_df['mbert_correct'].mean():.3f}", "| saved:", output_path)





Classifying rq1_eval...


Map:   0%|          | 0/166 [00:00<?, ? examples/s]

Map:   0%|          | 0/166 [00:00<?, ? examples/s]

mBERT average classification score: : 0.301 | saved: /content/data/results/rq1_matched_eval_baseline_mbert_predictions.csv
Classifying rq1_test...


Map:   0%|          | 0/166 [00:00<?, ? examples/s]

Map:   0%|          | 0/166 [00:00<?, ? examples/s]

... (9 outputs left out)

mBERT average classification score: : 0.325 | saved: /content/data/results/rq3_matched_eval_incorrect_mbert_predictions.csv
Classifying rq3_test...


Map:   0%|          | 0/83 [00:00<?, ? examples/s]

Map:   0%|          | 0/83 [00:00<?, ? examples/s]

mBERT average classification score: : 0.325 | saved: /content/data/results/rq3_matched_test_incorrect_mbert_predictions.csv


In [28]:
# Inspeciting the result since the average score is so low
for key, df in prediction_dfs.items():
    print("\n", key)
    print(df["mbert_prediction"].value_counts(normalize=True))


 rq1_eval
mbert_prediction
neutral     0.608434
sadness     0.102410
joy         0.102410
anger       0.096386
fear        0.066265
surprise    0.024096
Name: proportion, dtype: float64

 rq1_test
mbert_prediction
neutral     0.548193
joy         0.120482
sadness     0.114458
fear        0.096386
anger       0.090361
surprise    0.030120
Name: proportion, dtype: float64

 rq2_eval
mbert_prediction
neutral     0.542169
sadness     0.132530
joy         0.120482
anger       0.108434
fear        0.084337
surprise    0.012048
Name: proportion, dtype: float64

 rq2_test
mbert_prediction
neutral     0.506024
sadness     0.156627
anger       0.096386
fear        0.096386
joy         0.084337
surprise    0.060241
Name: proportion, dtype: float64

 rq3_eval
mbert_prediction
neutral     0.518072
sadness     0.144578
anger       0.120482
joy         0.096386
fear        0.084337
surprise    0.036145
Name: proportion, dtype: float64

 rq3_test
mbert_prediction
neutral     0.554217
joy         0.13

In [29]:
# Diagnostic only
# Evaluate mBERT directly on the original Chinese SMP text
# Nothing in this cell is saved to disk

def classify_chinese_source(frame):
    chinese_df = frame.copy()

    chinese_df["gold_shared_label"] = (
        chinese_df["label"].map(SMP_TO_SHARED)
    )

    inference_df = pd.DataFrame({
        "text": chinese_df["text"]
            .fillna("")
            .astype(str)
    })

    dataset = Dataset.from_pandas(
        inference_df,
        preserve_index=False,
    )

    tokenized_dataset = dataset.map(
        tokenize_function,
        batched=True,
    )

    prediction_output = inference_trainer.predict(
        tokenized_dataset
    )

    logits = prediction_output.predictions
    probabilities = torch.softmax(
        torch.tensor(logits),
        dim=1,
    ).numpy()

    prediction_ids = probabilities.argmax(axis=1)

    chinese_df["mbert_prediction"] = [
        id_to_label[int(idx)]
        for idx in prediction_ids
    ]

    chinese_df["mbert_correct"] = (
        chinese_df["mbert_prediction"]
        == chinese_df["gold_shared_label"]
    )

    return chinese_df


# Use RQ1 because it contains both slang + matched controls.
# Run eval and test separately.
for split in ["eval", "test"]:
    source_df = translation_dfs[f"rq1_{split}"]

    chinese_predictions = classify_chinese_source(
        source_df
    )

    y_true = (
        chinese_predictions["gold_shared_label"]
        .map(label_to_id)
        .to_numpy()
    )

    y_pred = (
        chinese_predictions["mbert_prediction"]
        .map(label_to_id)
        .to_numpy()
    )

    accuracy = accuracy_score(
        y_true,
        y_pred,
    )

    macro_f1 = f1_score(
        y_true,
        y_pred,
        labels=list(range(len(LABELS))),
        average="macro",
        zero_division=0,
    )

    print(f"\n{split.upper()} — ORIGINAL CHINESE")
    print(f"Accuracy: {accuracy:.3f}")
    print(f"Macro-F1: {macro_f1:.3f}")

Map:   0%|          | 0/166 [00:00<?, ? examples/s]


EVAL — ORIGINAL CHINESE
Accuracy: 0.699
Macro-F1: 0.642


Map:   0%|          | 0/166 [00:00<?, ? examples/s]


TEST — ORIGINAL CHINESE
Accuracy: 0.699
Macro-F1: 0.632


Notes:
- It seems like the majority of the translations get classified as `neutral`
- The accuracy on test and eval on the native chinese post is aroud 0.7 which is way closer to what I expected base on what we got for training in the previous notebook